# 垃圾识别  Garbage identify

In [ ]:
import sys
sys.path.append('/home/jetson/dofbot_ws/src/dofbot_garbage_yolov5')

### 导入头文件 Import head file

In [ ]:
#!/usr/bin/env python
# coding: utf-8
import Arm_Lib
import os
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from garbage_identify import garbage_identify
from dofbot_utils.fps import FPS
from dofbot_utils.robot_controller import Robot_Controller

### 创建实例,初始化参数 Create the instance and initialize the parameters

In [ ]:
robot = Robot_Controller()
robot.move_look_map()

In [ ]:
garbage = garbage_identify()
fps = FPS()
model = "General"

### 创建控件 Creating widget

In [ ]:
button_layout      = widgets.Layout(width='320px', height='60px', align_self='center')
output = widgets.Output()
# 退出
exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))

### 抓取控制  grab control

In [ ]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output: print(model)
exit_button.on_click(exit_button_Callback)

### 主程序 Main process

In [ ]:
def camera():
    # 打开摄像头 Open camera
    capture = cv.VideoCapture(0)
    capture.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    capture.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    # 当摄像头正常打开的情况下循环执行
    while capture.isOpened():
        try:
            _, img = capture.read()
            fps.update_fps()
            img, msg = garbage.garbage_run(img)
            if len(msg) > 0:
                for name, pos in msg.items():
                    print("name:", name)
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            fps.show_fps(img)
            imgbox.value = cv.imencode('.jpg', img)[1].tobytes()
        except Exception as e:
            capture.release()
            print(e)
            break

### 启动  Start

In [ ]:
# Please place the building block in the center of the cross (the picture is facing the mechanical arm)
# 请将积木块正放在十字中心(图片正对机械臂)
display(controls_box,output)
threading.Thread(target=camera, ).start()
